# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

Runtime → Change runtime type → **T4 GPU** → Save. Phần này: lấy repo, cài thư viện, nạp Kaggle Secrets,
mount Drive (lưu adapter/checkpoint), rồi chạy `pytest` để chắc công cụ của đề chạy đúng.

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L

# Mở notebook từ GitHub thì Colab đứng ở /content, chưa có scripts/ → clone repo và cd vào.
import os
REPO_URL = "https://github.com/SxAinsworth/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if not os.path.exists("scripts/score.py"):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
print("cwd:", os.getcwd())

In [ ]:
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print("Kaggle user:", os.environ['KAGGLE_USERNAME'])

In [ ]:
# Mount Google Drive để sao lưu adapter LoRA / projector / predictions sau mỗi lần chạy
# (session Colab miễn phí có thể bị ngắt giữa chừng).
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/lab_day10")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Sao lưu vào:", DRIVE_DIR)

import shutil

def backup(path):
    """Chép file/thư mục (đường dẫn tương đối trong repo) sang DRIVE_DIR."""
    src, dst = Path(path), DRIVE_DIR / path
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif src.exists():
        shutil.copy2(src, dst)

def restore(path):
    """Local chưa có mà Drive có (runtime mới sau khi bị ngắt) → chép về. Trả True nếu đã chép."""
    src, dst = DRIVE_DIR / path, Path(path)
    if dst.exists() or not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print("restore từ Drive:", path)
    return True

In [ ]:
# Kiểm tra công cụ của đề. Kỳ vọng: 68 passed, 1 skipped (test dữ liệu thật bị bỏ qua khi chưa tải dataset).
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải dataset Kaggle và chia split theo phiên quay (cố định, ai chạy cũng ra cùng kết quả).
# Bỏ qua nếu đã có data/ (chạy lại notebook sau khi session bị ngắt mà runtime còn giữ ổ đĩa).
# Không có Kaggle? Thay bằng: !python scripts/prepare_data.py --zip <đường_dẫn_zip>
import os
if not os.path.exists("data/splits/meta.json"):
    !python scripts/prepare_data.py --kaggle
else:
    print("Đã có data/splits/meta.json, bỏ qua bước tải.")

In [ ]:
# Thống kê từng split + kiểm tra rò rỉ phiên quay (số liệu cho câu 6 của REPORT).
import json
from collections import Counter
from scripts.labcommon import CLASSES, read_jsonl

SPLITS = ("train", "val", "test")
data = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in SPLITS}
meta = json.load(open("data/splits/meta.json", encoding="utf-8"))
DATASET_VERSION = meta["dataset_version"]
print("dataset_version:", DATASET_VERSION)
assert DATASET_VERSION.startswith("d10-"), DATASET_VERSION

# Bảng chuẩn của đề (README mục 3): ảnh, pothole, crack, manhole
EXPECTED = {"train": (1422, 1074, 1750, 691), "val": (276, 66, 470, 103), "test": (311, 121, 299, 163)}
print(f"{'split':6s} {'ảnh':>5s} {'sessions':>8s} " + " ".join(f"{c:>8s}" for c in CLASSES) + f" {'box/ảnh':>8s} {'ảnh rỗng':>8s}")
for s in SPLITS:
    rows = data[s]
    box = Counter(o["label"] for r in rows for o in r["objects"])
    n_box = sum(box.values())
    n_empty = sum(1 for r in rows if not r["objects"])
    n_sess = len({r["session"] for r in rows})
    print(f"{s:6s} {len(rows):5d} {n_sess:8d} " + " ".join(f"{box[c]:8d}" for c in CLASSES)
          + f" {n_box / len(rows):8.2f} {n_empty:8d}")
    got = (len(rows), *(box[c] for c in CLASSES))
    assert got == EXPECTED[s], f"{s}: {got} != {EXPECTED[s]}"

# Không session nào nằm ở 2 split
sess = {s: {r["session"] for r in data[s]} for s in SPLITS}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    inter = sess[a] & sess[b]
    assert not inter, f"Rò rỉ phiên quay {a}/{b}: {sorted(inter)[:5]}"
print("OK: số ảnh/box khớp bảng của đề; không session nào xuất hiện ở hai split.")

# Kích thước box: diện tích tương đối (w*h) theo lớp, trên train (dùng cho câu 9)
import statistics
for c in CLASSES:
    areas = [o["yolo"][2] * o["yolo"][3] for r in data["train"] for o in r["objects"] if o["label"] == c]
    print(f"  {c:8s} diện tích trung vị = {100 * statistics.median(areas):.2f}% khung hình")

In [ ]:
# Minh hoạ vì sao KHÔNG chia ngẫu nhiên: nếu chia theo ảnh (cùng tỉ lệ ~15% test),
# bao nhiêu ảnh test có "anh em" cùng phiên quay nằm trong train? (số liệu cho câu 6)
import random
all_rows = [r for s in SPLITS for r in data[s]]
rng = random.Random(0)
shuffled = all_rows[:]
rng.shuffle(shuffled)
n_test = len(data["test"])
rand_test, rand_train = shuffled[:n_test], shuffled[n_test:]
train_sess = {r["session"] for r in rand_train}
leaked = sum(1 for r in rand_test if r["session"] in train_sess)
print(f"Chia ngẫu nhiên: {leaked}/{n_test} ảnh test ({100 * leaked / n_test:.1f}%) có cùng phiên quay với ảnh train.")
print(f"Chia theo phiên (prepare_data.py): 0/{n_test} ảnh test.")
print("Số phiên quay:", len({r['session'] for r in all_rows}), "| ảnh/phiên trung bình:",
      round(len(all_rows) / len({r['session'] for r in all_rows}), 1))

Vẽ 6 ảnh train kèm box gốc (YOLO → pixel) để chắc hiểu nhãn. Ảnh được lưu vào `report/figures/data_samples.png`.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def draw_yolo(img, objects, width=3):
    """Vẽ box YOLO (xc, yc, w, h chuẩn hoá [0,1]) lên bản sao của ảnh."""
    img = img.copy()
    d = ImageDraw.Draw(img)
    W, H = img.size
    for o in objects:
        xc, yc, w, h = o["yolo"]
        box = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
        d.rectangle(box, outline=COLORS[o["label"]], width=width)
        d.text((box[0] + 2, box[1] + 2), o["label"], fill=COLORS[o["label"]])
    return img

rng = random.Random(10)
# Lấy mẫu có đủ cả 3 lớp: 2 ảnh có pothole, 2 có crack, 2 có manhole
samples = []
for c in CLASSES:
    pool = [r for r in data["train"] if any(o["label"] == c for o in r["objects"]) and r not in samples]
    samples += rng.sample(pool, 2)

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, samples):
    img = Image.open(Path("data") / r["image_path"]).convert("RGB")
    ax.imshow(draw_yolo(img, r["objects"]))
    ax.set_title(f"{r['image_id']}\n{r['session']} · {len(r['objects'])} box", fontsize=9)
    ax.axis("off")
fig.suptitle("Nhãn gốc (đỏ=pothole, vàng=crack, xanh=manhole)")
plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("report/figures/data_samples.png", dpi=80, bbox_inches="tight")
plt.show()
print("Kích thước ảnh (W, H):", Counter((r["width"], r["height"]) for r in all_rows))

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

def yolo_to_box(label, yolo):
    """YOLO (xc, yc, w, h chuẩn hoá [0,1]) → {"label", "bbox_2d": [x1,y1,x2,y2]} nguyên trên thang 0–1000."""
    xc, yc, w, h = yolo
    clamp = lambda v: int(round(max(0.0, min(1000.0, v * 1000))))
    x1, y1 = clamp(xc - w / 2), clamp(yc - h / 2)
    x2, y2 = clamp(xc + w / 2), clamp(yc + h / 2)
    # Box rất nhỏ có thể bị làm tròn thành suy biến → parse_output của trọng tài sẽ coi là JSON hỏng.
    if x2 <= x1:
        x1, x2 = (x1, x1 + 1) if x1 < 1000 else (999, 1000)
    if y2 <= y1:
        y1, y2 = (y1, y1 + 1) if y1 < 1000 else (999, 1000)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """Danh sách object → chuỗi JSON đáp án. Sắp theo (y1, x1): trên→dưới, trái→phải để model học một thứ tự ổn định."""
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))   # "[]" nếu ảnh không có hư hại

PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

r0 = data["train"][0]
print(r0["image_id"], "→", objects_to_answer(r0["objects"]))
print("ảnh rỗng →", objects_to_answer([]))

In [ ]:
# Kiểm tra 1: JSON 0–1000 → pixel, vẽ chồng lên box gốc (YOLO). Hai box phải trùng nhau.
def answer_to_pixels(answer, W, H):
    return [(b["label"], [b["bbox_2d"][0] * W / 1000, b["bbox_2d"][1] * H / 1000,
                          b["bbox_2d"][2] * W / 1000, b["bbox_2d"][3] * H / 1000]) for b in json.loads(answer)]

r = max(data["train"], key=lambda r: len(r["objects"]))      # ảnh nhiều box nhất
img = Image.open(Path("data") / r["image_path"]).convert("RGB")
W, H = img.size
gt_img = draw_yolo(img, r["objects"], width=5)
d = ImageDraw.Draw(gt_img)
pix = answer_to_pixels(objects_to_answer(r["objects"]), W, H)
for lab, b in pix:
    d.rectangle(b, outline="white", width=1)
plt.figure(figsize=(10, 6)); plt.imshow(gt_img); plt.axis("off")
plt.title(f"{r['image_id']}: màu = box YOLO gốc, trắng = JSON 0–1000 quy ngược về pixel"); plt.show()

# Sai số quy đổi lớn nhất trên toàn bộ train (pixel), so từng object: YOLO → pixel vs YOLO → 0–1000 → pixel.
# Kỳ vọng ≤ 0.5/1000 × 640 ≈ 0.32 px (trừ box quá nhỏ bị nới thêm 1 đơn vị).
err = 0.0
for r in data["train"]:
    for o in r["objects"]:
        xc, yc, w, h = o["yolo"]
        ref = [max(0, xc - w / 2) * W, max(0, yc - h / 2) * H, min(1, xc + w / 2) * W, min(1, yc + h / 2) * H]
        b = yolo_to_box(o["label"], o["yolo"])["bbox_2d"]
        got = [b[0] * W / 1000, b[1] * H / 1000, b[2] * W / 1000, b[3] * H / 1000]
        err = max(err, max(abs(u - v) for u, v in zip(ref, got)))
print(f"Sai số quy đổi lớn nhất: {err:.2f} px")

In [ ]:
# Kiểm tra 2 (dùng cho câu 3): đưa chính đáp án đúng vào trọng tài.
#  - đáp án 0–1000, chấm --coord norm1000  → F1 ≈ 1.0 (chỉ box cực nhỏ có thể lệch IoU vì làm tròn)
#  - cùng đáp án, chấm --coord pixel       → trọng tài hiểu số là pixel → lệch thang
#  - đáp án pixel,  chấm --coord norm1000  → lệch thang theo chiều ngược lại
from scripts.score import evaluate

def objects_to_answer_pixel(objects, W=640, H=360):
    boxes = []
    for o in objects:
        b = yolo_to_box(o["label"], o["yolo"])["bbox_2d"]
        boxes.append({"label": o["label"], "bbox_2d": [round(b[0] * W / 1000), round(b[1] * H / 1000),
                                                      round(b[2] * W / 1000), round(b[3] * H / 1000)]})
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

ents = data["val"]
gt_norm = {r["image_id"]: objects_to_answer(r["objects"]) for r in ents}
gt_pix = {r["image_id"]: objects_to_answer_pixel(r["objects"]) for r in ents}
for name, preds, coord in (("norm1000 → --coord norm1000", gt_norm, "norm1000"),
                           ("norm1000 → --coord pixel   ", gt_norm, "pixel"),
                           ("pixel    → --coord pixel   ", gt_pix, "pixel"),
                           ("pixel    → --coord norm1000", gt_pix, "norm1000")):
    res = evaluate(ents, preds, coord=coord, n_boot=200)
    print(f"{name}: valid={res['parse_valid_rate']:.3f}  F1={res['micro']['f1']:.3f}  meanIoU={res['mean_iou']:.3f}")

## 3. E0: zero-shot

In [ ]:
# Nạp model gốc ở 4-bit NF4. ViT (kể cả projector) và lm_head giữ fp16:
# nếu lượng tử hoá cả "visual", trọng số merger thành 4-bit → không huấn luyện đầy đủ được ở E2/E3.
# dtype=float16 vì T4 không hỗ trợ bf16 hiệu quả (config gốc của Qwen3-VL là bf16).
import gc, time
import torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from scripts.labcommon import write_jsonl

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"])
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quantization, dtype=torch.float16, device_map={"": 0})
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = "left"      # sinh theo batch: pad bên trái để token cuối là prompt
    return model, processor

def load_image(row):
    return Image.open(Path("data") / row["image_path"]).convert("RGB")

def user_message(image, prompt):
    return {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}

def run_inference(model, processor, rows, out_path, prompt=PROMPT, batch_size=4, max_new_tokens=256):
    """Suy luận greedy, ghi {"image_id","raw"} vào out_path. Chạy tiếp được nếu bị ngắt (bỏ qua ảnh đã có)."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    restore(str(out_path))
    done = {r["image_id"] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r["image_id"] not in done]
    print(f"{out_path}: đã có {len(done)}, còn {len(todo)} ảnh")
    model.eval()
    t0 = time.time()
    with open(out_path, "a", encoding="utf-8") as f:
        for b in tqdm(range(0, len(todo), batch_size)):
            batch = todo[b:b + batch_size]
            msgs = [[user_message(load_image(r), prompt)] for r in batch]
            inputs = processor.apply_chat_template(
                msgs, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True).to(model.device)
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for r, t in zip(batch, texts):
                f.write(json.dumps({"image_id": r["image_id"], "raw": t}, ensure_ascii=False) + "\n")
            f.flush()
            if (b // batch_size) % 20 == 19:
                backup(str(out_path))
    backup(str(out_path))
    print(f"xong trong {(time.time() - t0) / 60:.1f} phút")

In [ ]:
# E0: zero-shot trên toàn bộ 311 ảnh test → work/e0.jsonl
model, processor = load_base_model()
print(f"VRAM sau khi nạp model: {torch.cuda.memory_allocated() / 2**30:.2f} GB")
run_inference(model, processor, data["test"], "work/e0.jsonl")

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
backup("report/evidence/e0.json")

In [ ]:
# Phân tích output E0 (câu 2): parse_valid_rate, các kiểu lỗi, 5 output thô
import re
from collections import Counter
from scripts.score import parse_output

e0 = read_jsonl("work/e0.jsonl")
gt_by_id = {r["image_id"]: r for r in data["test"]}
kinds = Counter()
for p in e0:
    raw, parsed = p["raw"], parse_output(p["raw"])
    if "```" in raw:
        kinds["bọc ```json```"] += 1
    if parsed is None:
        kinds["JSON hỏng / không parse được"] += 1
        continue
    if not raw.strip().startswith(("[", "```")):
        kinds["có chữ thừa ngoài JSON"] += 1
    if not parsed and gt_by_id[p["image_id"]]["objects"]:
        kinds["trả [] dù ảnh có hư hại"] += 1
    if any(b["label"] not in CLASSES for b in parsed):
        kinds["nhãn ngoài 3 lớp"] += 1
    if any(max(b["bbox"]) > 1000 for b in parsed):
        kinds["toạ độ > 1000 (thang pixel/sai thang)"] += 1
    if len(parsed) > 3 * max(len(gt_by_id[p["image_id"]]["objects"]), 1):
        kinds["bịa nhiều box (> 3× số box thật)"] += 1
print(f"{len(e0)} output. Phân loại lỗi (một ảnh có thể rơi vào nhiều loại):")
for k, v in kinds.most_common():
    print(f"  {v:4d}  {k}")
print("Nhãn model dùng:", Counter(b["label"] for p in e0 for b in (parse_output(p["raw"]) or [])).most_common(8))

for p in e0[:5]:
    print("-" * 80)
    print(p["image_id"], "| GT:", objects_to_answer(gt_by_id[p["image_id"]]["objects"]))
    print("RAW:", p["raw"][:400])

In [ ]:
# Giải phóng VRAM trước khi sang phần huấn luyện
del model
gc.collect(); torch.cuda.empty_cache()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9